# Fine-tune for Math Reasoning
#### Dataset: Use the "gsm8k" dataset from Hugging Face :
https://huggingface.co/datasets/openai/gsm8k


**Trained on openai/gsm8k dataset**

First we check the GPU version available in the environment and install specific dependencies that are compatible with the detected GPU to prevent version conflicts.

In [ ]:
%%capture
#magic command : Do all this work silently. Don't show me the messy output unless there is an error. like downloading , installing etc...
import torch #imports pyTorch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" #Downloads and installs unsloth directly from github
if major_version >= 8:
    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
    #installs flash-attn (Flash Attention) which is a cutting-edge speed booster
else:
    !pip install --no-deps xformers trl peft accelerate bitsandbytes
    #skips flash-attn because the older card physically cannot handle it. It installs the standard tools (xformers) instead.
pass

In [ ]:
from unsloth import FastLanguageModel #fetch fastLanguageModel which is a optimized special layer of code wrapped around standard tools to make it faster
import torch #import pyTorch handles the heavy math (matrix multiplication)
max_seq_length = 2048 # Attention span/ context window (read and hold which is approx 1500 words or about 3-4 pages)
dtype = None #datatype = none whihc Auto-detect the best math precision for my specific hardware. Which prevents compatibility errors
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
#The above code is of quantization and if we use 16 bit precision then it uses the 16GB full storage provided by colab and then we will left with 0GB to train the model
#The model shrinks from 16 GB down to about 5.5 GB
# we are not doenloading the below listed models just listing them so we can use any of them as per need.

fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",
    "unsloth/gemma-7b-it-bnb-4bit",
    "unsloth/gemma-2b-bnb-4bit",
    "unsloth/gemma-2b-it-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit"
]
# Below code goes to internet and downloads the model and the tokenizer of the model listed , also max_seq_length = 2048 , dtype = none and quantized version
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit", # Llama-3 70b also works (just change the model name)
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
    #Some models (like the original Llama 2 from Meta) are "Gated" or private.
    #We need to create an account, accept a license agreement, and get a password (Token) to download them.
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.10: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Next, we integrate LoRA adapters into our model, which allows us to efficiently update just a fraction of the model's parameters, enhancing training speed and reducing computational load.

In [ ]:
#PEFT (Parameter-Efficient Fine-Tuning): attaching small, trainable "adapters"
# Low-Rank Adaptation(LoRA) is a PEFT technique used to adapt large, pre trained models  to specific tasks or domains without retraining entire base model.(Broader term for PEFT)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128 which is size of the "Post-it note"/adapters
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],

    # The terms q_proj (Query), k_proj (Key), v_proj (Value) are parts of the Self-Attention mechanism—the part of the brain that understands relationships between words.
    lora_alpha = 16, #learning scaling factor is generally set to r , If you set alpha drastically higher, the model might ignore its original knowledge entirely.
    lora_dropout = 0, #Don't randomly ignore data during training
    bias = "none", #Do not train the bias parameters , so lightweight and more faster training
    use_gradient_checkpointing = "unsloth", #Use Unsloth's special memory-saving trick
    random_state = 3407, #we ensure that if you run this code today and I run it tomorrow, we get the exact same result. It makes your experiment reproducible.
    use_rslora = False, #Do not use Rank-Stabilized LoRA
    loftq_config = None, #Do not use LoRA-Fine-Tuning Quantization. We don't need as we are already using the load_in_4bit
)

Unsloth 2025.12.10 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


# Data Prep
We now use the openai/gsm8k,which is a dataset of 8.5K high quality linguistically diverse grade school math word problems. The dataset was created to support the task of question answering on basic mathematical problems that require multi-step reasoning.. We can replace this code section with own data prep.

Then, we define a system prompt that formats tasks into instructions, inputs, and responses, and apply it to a dataset to prepare our inputs and outputs for the model, with an EOS token to signal completion.

In [ ]:
from datasets import load_dataset

# Load GSM8K train split
dataset = load_dataset("openai/gsm8k", "main", split="train")

# Simple math reasoning prompt
math_prompt = """Below is a math word problem. Solve it step by step and provide the final answer.

### Problem:
{question}

### Solution:
{answer}"""

EOS_TOKEN = tokenizer.eos_token


def formatting_prompts_func(examples):
    texts = []

    for q, a in zip(examples["question"], examples["answer"]):
        text = math_prompt.format(question=q, answer=a) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}


# Apply formatting — outputs dataset["text"]
dataset = dataset.map(formatting_prompts_func, batched=True)


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

<a name="Train"></a>
### Train the model
- We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.
- At this stage, we're configuring our model's training setup, where we define things like batch size and learning rate, to teach our model effectively with the data we have prepared.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,

    # KEY SETTING FOR REASONING TASKS
    train_on_inputs = False,                                            # Loss is NOT applied on the question
    packing = False,                                                   # Better for reasoning quality
   # KEY SETTING FOR REASONING TASKS
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,                                      # Effective batch size = 8
        warmup_steps = 20,                                                    # Slightly higher warmup = more stable
        max_steps = 935,
        learning_rate = 2e-4,                                                    # Good for LoRA
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_steps = 200,                                                   # Save checkpoints during training
        save_total_limit = 2,                                               # Keep disk usage low
    ),
)


In [ ]:
#@title Show current memory stats
import torch

gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

print(f"GPU Detected = {gpu_stats.name}")
print(f"Total GPU Memory = {max_memory} GB")
print(f"Memory Reserved Before Training = {start_gpu_memory} GB")

GPU Detected = Tesla T4
Total GPU Memory = 14.741 GB
Memory Reserved Before Training = 6.973 GB


In [ ]:
# We're now kicking off the actual training of our model, which will spit out some statistics showing us how well it learns
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,1.551700
10,1.376300
15,1.239300
20,1.001300
25,0.963800
30,0.975800
35,0.899300
40,0.853100
45,0.875300
50,0.888900


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
train/global_step,▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▄█▄▅▃▄▃▃▂▂▂▂▂▂▁▂▁▁▂▂▂▂▂▂▁▂▃▃▃▁▂▂▂▂▁▂▂▂▂▂
train/learning_rate,▄▆███▇▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▂▂▁
train/loss,█▆▃▃▂▂▂▂▂▂▂▂▂▁▂▂▂▁▁▂▂▂▂▁▂▂▂▁▂▂▁▂▁▁▂▁▂▁▂▁
total_flos,3.920664188819866e+16
train/epoch,0.53519
train/global_step,500
train/grad_norm,0.34331
train/learning_rate,0.0
train/loss,0.8238


In [ ]:
# --- MEMORY + TRAINING TIME REPORT --- for 500 steps

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"Training Runtime  = {trainer_stats.metrics['train_runtime']} seconds")
print(f"Training Runtime  = {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes")
print(f"Peak GPU Reserved = {used_memory} GB")
print(f"LoRA Training GPU Usage = {used_memory_for_lora} GB")
print(f"Peak GPU Utilization = {used_percentage} %")
print(f"LoRA Training Utilization = {lora_percentage} %")

# switch model to inference mode (important)
FastLanguageModel.for_inference(model)


Training Runtime  = 3436.1072 seconds
Training Runtime  = 57.27 minutes
Peak GPU Reserved = 7.289 GB
LoRA Training GPU Usage = 0.316 GB
Peak GPU Utilization = 49.447 %
LoRA Training Utilization = 2.144 %


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
# Resume training: This keeps your old logs safe above and starts a new log table below
trainer_stats = trainer.train(resume_from_checkpoint = True)

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 935
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
505,0.834500
510,0.819200
515,0.773200
520,0.798200
525,0.832500
530,0.786600
535,0.796900
540,0.779800
545,0.780300
550,0.867300


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▄▄▂▄▄▁▄▄▃▇▃▅▂▃▄▁▂▁▃▆▃▂▆▆▁▂█▆▇█▆█▄▃▅▄▄▂▂▄
train/learning_rate,████▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/loss,▅▄▃▆▂█▅▂▆▄▄▄▅▅▅▃▅▃▅▃▅█▃▇█▆▄▄▄▁▄▅▇▄▄▃▂▂▂▅
total_flos,7.274335225184256e+16
train/epoch,1
train/global_step,935
train/grad_norm,0.74841
train/learning_rate,0.0
train/loss,0.7309


In [ ]:
# --- FINAL MEMORY + TRAINING TIME REPORT --- FOR left steps (1 epoch completed combined)

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"Training Runtime  = {trainer_stats.metrics['train_runtime']} seconds")
print(f"Training Runtime  = {round(trainer_stats.metrics['train_runtime']/60, 2)} minutes")
print(f"Peak GPU Reserved = {used_memory} GB")
print(f"LoRA Training GPU Usage = {used_memory_for_lora} GB")
print(f"Peak GPU Utilization = {used_percentage} %")
print(f"LoRA Training Utilization = {lora_percentage} %")

# switch model to inference mode (important)
FastLanguageModel.for_inference(model)


Training Runtime  = 2925.6306 seconds
Training Runtime  = 48.76 minutes
Peak GPU Reserved = 7.301 GB
LoRA Training GPU Usage = 0.328 GB
Peak GPU Utilization = 49.529 %
LoRA Training Utilization = 2.225 %


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

### Running test on 50 Examples for Evaluation of Accuracy

##### Futher Detailed test done with

In [ ]:
from tqdm import tqdm
import re

# Load the test set (data the model has NEVER seen)
test_dataset = load_dataset("openai/gsm8k", "main", split="test")

def extract_number(text):
    # Finds the number after "####" which is the GSM8K standard for answers
    pattern = r"####\s*(-?\d+\.?\d*)"
    match = re.search(pattern, text)
    return float(match.group(1)) if match else None

# Test on 50 examples
sample_size = 50
correct_count = 0

print(f"Running evaluation on {sample_size} test examples...")

for i in tqdm(range(sample_size)):
    problem = test_dataset[i]['question']
    true_answer = extract_number(test_dataset[i]['answer'])

    # Format prompt exactly like training
    prompt = f"Below is a math word problem. Solve it step by step and provide the final answer.\n\n### Problem:\n{problem}\n\n### Solution:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate answer
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    text_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Check if model got it right
    model_answer = extract_number(text_output)

    if model_answer is not None and true_answer is not None:
        if abs(model_answer - true_answer) < 1e-6: # precise comparison
            correct_count += 1

print(f"Final Accuracy: {(correct_count/sample_size)*100}%")

Running evaluation on 50 test examples...


100%|██████████| 50/50 [06:16<00:00,  7.53s/it]

Final Accuracy: 66.0%


# INFERENCE AND ERROR ANALYSIS

In [ ]:
def solve_math(question):
    prompt = f"""Below is a math word problem. Solve it step by step and provide the final answer.

### Problem:
{question}

### Solution:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens = 256,
        temperature = 0.2,     # deterministic reasoning
        top_p = 0.9,
        do_sample = True,
        use_cache = True,
        eos_token_id = tokenizer.eos_token_id,
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)


In [ ]:
response = solve_math(
    "Tom has 5 apples and buys 7 more. He then gives 4 to his friend. How many apples does he have now?"
)

print(response)


Below is a math word problem. Solve it step by step and provide the final answer.

### Problem:
Tom has 5 apples and buys 7 more. He then gives 4 to his friend. How many apples does he have now?

### Solution:
He has 5 + 7 = <<5+7=12>>12 apples.
He gives 4 to his friend, so he has 12 - 4 = <<12-4=8>>8 apples.
#### 8


In [ ]:
from transformers import TextStreamer

def solve_math_stream(question):
    prompt = f"""Below is a math word problem. Solve it step by step and provide the final answer.

### Problem:
{question}

### Solution:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    streamer = TextStreamer(tokenizer)

    _ = model.generate(
        **inputs,
        streamer = streamer,
        max_new_tokens = 256,
        temperature = 0.2,
        top_p = 0.9,
        do_sample = True,
        eos_token_id = tokenizer.eos_token_id,
        use_cache = True,
    )


In [ ]:
solve_math_stream("Sarah has twice as many marbles as Jim. Jim has 6. How many does Sarah have?")


<|begin_of_text|>Below is a math word problem. Solve it step by step and provide the final answer.

### Problem:
Sarah has twice as many marbles as Jim. Jim has 6. How many does Sarah have?

### Solution:
Sarah has 6*2=<<6*2=12>>12 marbles
#### 12<|end_of_text|>


In [ ]:
# --- SAVE THE MODEL ---
# This saves only the LoRA adapters (lightweight, ~100MB)
model.save_pretrained("fine_tuned_gsm8k_adapters")
tokenizer.save_pretrained("fine_tuned_gsm8k_adapters")

print("Adapters saved successfully!")

Adapters saved successfully!


In [ ]:
from google.colab import files

# Zip the folder first (easier to download 1 file than many)
!zip -r fine_tuned_gsm8k.zip fine_tuned_gsm8k_adapters

# Trigger the download
files.download('fine_tuned_gsm8k.zip')

  adding: fine_tuned_gsm8k_adapters/ (stored 0%)
  adding: fine_tuned_gsm8k_adapters/tokenizer.json (deflated 85%)
  adding: fine_tuned_gsm8k_adapters/special_tokens_map.json (deflated 71%)
  adding: fine_tuned_gsm8k_adapters/tokenizer_config.json (deflated 96%)
  adding: fine_tuned_gsm8k_adapters/adapter_config.json (deflated 58%)
  adding: fine_tuned_gsm8k_adapters/adapter_model.safetensors (deflated 7%)
  adding: fine_tuned_gsm8k_adapters/README.md (deflated 65%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# --- ERROR ANALYSIS ---
print("Running Error Analysis on 5 Failed Examples...")

error_count = 0
for i in range(len(test_dataset)):
    if error_count >= 3: break # Only look at 3 failures

    question = test_dataset[i]['question']
    true_answer_text = test_dataset[i]['answer']
    true_num = extract_number(true_answer_text)

    # Generate
    inputs = tokenizer(
        f"Below is a math word problem. Solve it step by step and provide the final answer.\n\n### Problem:\n{question}\n\n### Solution:\n",
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    model_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    model_num = extract_number(model_output)

    # If Wrong
    if model_num is None or (true_num is not None and abs(model_num - true_num) > 1e-6):
        error_count += 1
        print(f"\n--- FAILURE CASE {error_count} ---")
        print(f"QUESTION: {question}")
        print(f"TRUE ANSWER: {true_num}")
        print(f"MODEL THOUGHT: {model_num}")
        print(f"FULL GENERATION:\n{model_output.split('### Solution:')[1].strip()[:200]}...") # Print first 200 chars of solution

Running Error Analysis on 5 Failed Examples...

--- FAILURE CASE 1 ---
QUESTION: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?
TRUE ANSWER: 70000.0
MODEL THOUGHT: -38000.0
FULL GENERATION:
He spent 80000+50000=$<<80000+50000=130000>>130,000
The value increased by 80000*.15=$<<80000*.15=12000>>12,000
So the value of the house is now 80000+12000=$<<80000+12000=92000>>92,000
So he made 920...

--- FAILURE CASE 2 ---
QUESTION: Kylar went to the store to buy glasses for his new apartment. One glass costs $5, but every second glass costs only 60% of the price. Kylar wants to buy 16 glasses. How much does he need to pay for them?
TRUE ANSWER: 64.0
MODEL THOUGHT: 125.0
FULL GENERATION:
The price of the second glass is 5 * 60/100 = $<<5*60/100=3>>3.
So the price of 16 glasses is 16 * 5 = $<<16*5=80>>80.
The price of 15 glasses is 15 * 3 = $<<15*3=45>>45.
The to

In [ ]:
# --- INFERENCE PLAYGROUND ---
def ask_math_question(question_text):
    prompt = f"Below is a math word problem. Solve it step by step and provide the final answer.\n\n### Problem:\n{question_text}\n\n### Solution:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Solution:")[1])

# Try your own question!
ask_math_question("If I have 30 chocolates and I eat 2 every day, how many will I have left after 1 week?")


I will eat 2 * 7 = <<2*7=14>>14 chocolates in 1 week.
So, I will have 30 - 14 = <<30-14=16>>16 chocolates left.
#### 16
